# 02 · Build the matrix from the information engine

## Question

Where do the four entries of $J_{ij}$ come from physically?

## Concept

**Physics / quantum view.** An optical tweezer uses focused light to confine the bead. Thermal collisions cause Brownian motion. The simulator models it with a Langevin equation, which combines deterministic forces and random forcing. Stochastic thermodynamics tracks quantities such as work along these fluctuating trajectories.

$$(\rho_i,\pi_j)\ \longrightarrow\ \text{fixed-policy IE run}\ \longrightarrow\ \text{scalar performance}\ \longrightarrow\ J_{ij}.$$

Here $\rho_i$ denotes operating regime $i$, $\pi_j$ denotes policy $j$, and $J_{ij}$ is the mean performance. Work and heat along fluctuating trajectories are part of stochastic thermodynamics [1].

The plant simulates a bead in an optical trap. The measurement block adds sensor noise. The estimator and controller use the selected policy to command trap moves. The two regimes vary the actual sensor-noise RMS from 2 nm to 6 nm. Each policy keeps its own estimator and controller parameters for the entire run.

The existing simulator builder applies the policy at time zero and disables supervisor switching. The evaluator checks the policy history after each trial. Measure signed trap power and gravitational power over the same interval after the 0.1 s warm-up. The offline runner stores the cost $C=P_\mathrm{trap}-P_\mathrm{grav}$ in $k_BT/\mathrm{s}$. Here $P_\mathrm{trap}$ is signed trap power and $P_\mathrm{grav}$ is gravitational power. $k_B$ is the Boltzmann constant and $T$ the bath temperature. The teaching convention is performance $J=-C$, so higher is better.

Notebook 01 used illustrative scores in arbitrary units. The following runs produce **IE-derived** scores in physical units. Run from a kernel inside the checkout. The five-seed demonstration takes longer than the other notebooks.

## Check / exercise

Before running, predict whether the robust policy has a larger advantage at 2 nm or 6 nm. Explain why reusing the same seeds lets you compare policies in pairs.

## Worked example

In [1]:
from pathlib import Path
from statistics import mean, stdev
import sys

cwd = Path.cwd()
root = next(path for path in (cwd, *cwd.parents)
            if (path / "ie" / "run_synthesis_offline.py").is_file())
sys.path.insert(0, str(root / "ie"))
try:
    from synthesis.evaluate import evaluate_trial
    from synthesis.regimes import REGIMES
    from synthesis.synthesizer import QSynthesizer
finally:
    sys.path.pop(0)

policies = QSynthesizer().synthesize()
seeds = tuple(range(42, 47))
duration_s, warmup_s = 1.0, 0.1
trials = {}
for regime, sigma_m in REGIMES.items():
    for policy in policies:
        scores = []
        for seed in seeds:
            result = evaluate_trial(policies, policy, sigma_m, seed, duration_s, warmup_s)
            scores.append(-result["cost_kbt_per_s"])
        trials[regime, policy] = scores
        print(f"Completed {regime} / {policy}, σ={sigma_m * 1e9:g} nm, n={len(scores)}")

Completed baseline_noise / nominal, σ=2 nm, n=5


Completed baseline_noise / robust, σ=2 nm, n=5


Completed elevated_noise / nominal, σ=6 nm, n=5


Completed elevated_noise / robust, σ=6 nm, n=5


## Result

Report the sample standard deviation (spread among runs) and standard error of the mean (estimated precision of the mean). For the policy comparison, calculate robust minus nominal for each matched seed before averaging.

In [2]:
from math import isfinite, sqrt

J = [[mean(trials[regime, policy]) for policy in policies] for regime in REGIMES]
print("Regime / policy                 Mean       SD       SE      [kBT/s]")
for (regime, policy), scores in trials.items():
    sd = stdev(scores)
    print(f"{regime:16} {policy:8} {mean(scores):8.2f} {sd:8.2f} {sd / sqrt(len(seeds)):8.2f}")
assert len(trials) == 4 and all(len(scores) == len(seeds) for scores in trials.values())
assert all(isfinite(value) for row in J for value in row)
for regime in REGIMES:
    differences = [robust - nominal for nominal, robust in
                   zip(trials[regime, "nominal"], trials[regime, "robust"], strict=True)]
    print(f"{regime}: robust − nominal = {mean(differences):+.2f} kBT/s "
          f"(paired SE {stdev(differences) / sqrt(len(seeds)):.2f})")

Regime / policy                 Mean       SD       SE      [kBT/s]
baseline_noise   nominal     72.07     8.48     3.79
baseline_noise   robust      70.52     8.36     3.74
elevated_noise   nominal     63.83     8.87     3.96
elevated_noise   robust      65.88     8.30     3.71
baseline_noise: robust − nominal = -1.56 kBT/s (paired SE 0.43)
elevated_noise: robust − nominal = +2.05 kBT/s (paired SE 0.57)


**Exercise check.** Compare the two paired differences above. Pairing keeps the randomness inputs matched across policies so the comparison can focus on the policy change. Five seeds give a pilot estimate.

## Takeaway

Each matrix entry is an estimate from repeated runs under one fixed regime and policy.

## Next step

[Notebook 03](03_policy_problem_to_qubo.ipynb) keeps the illustrative matrix as a stable teaching reference while it introduces a binary encoding. For a reusable measured artifact, run the existing `python ie/run_synthesis_offline.py` from the repository root. It writes `trials.csv` and `instance.json` under the Git-ignored runs directory.

### Presentation asset

Use the regime–policy → simulator → score → matrix diagram and the four means with paired policy differences.

MkDocs topic for this notebook: `docs/ie-cost-matrix.md`.

### References

[1] U. Seifert, [Stochastic thermodynamics, fluctuation theorems and molecular machines](https://doi.org/10.1088/0034-4885/75/12/126001), 2012.